# Классификаторы кухни 2

**ML4 · Classification** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

Во втором уроке по классификации вы изучите дополнительные способы классификации числовых данных. Вы также узнаете о последствиях выбора одного классификатора вместо другого.


### Предварительные требования

Мы предполагаем, что вы прошли предыдущие уроки и у вас есть очищенный набор данных в папке `data` с именем _cleaned_cuisines.csv_ в корне этой папки с 4 уроками.

### Подготовка

Мы загрузили ваш файл _notebook.ipynb_ с очищенным набором данных и разделили его на датафреймы X и y, готовые к процессу построения модели.



## Карта классификации

Ранее вы узнали о различных вариантах классификации данных, используя шпаргалку Microsoft. Scikit-learn предлагает похожую, но более детальную шпаргалку, которая поможет сузить выбор оценщиков (другой термин для классификаторов):

![ML Map from Scikit-learn](ms-assets/ml/map.e963a6a51349425a.webp)
> Подсказка: [посетите эту карту онлайн](https://scikit-learn.org/stable/tutorial/machine_learning_map/) и переходите по пути, чтобы читать документацию.

### План

Эта карта очень полезна, когда у вас есть четкое понимание ваших данных, поскольку вы можете «идти» по ее путям к принятию решения:

- У нас более 50 образцов
- Мы хотим предсказать категорию
- У нас есть размеченные данные
- У нас менее 100 тысяч образцов
- ✨ Мы можем выбрать Linear SVC
- Если это не сработает, так как у нас числовые данные
    - Мы можем попробовать ✨ KNeighbors Classifier 
      - Если это не сработает, попробуйте ✨ SVC и ✨ Ensemble Classifiers

Это очень полезный путь для следования.



## Упражнение - разделение данных

Следуя этому пути, начнем с импорта нужных библиотек.

1. Импортируйте необходимые библиотеки:

    ```python
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import accuracy_score,precision_score,confusion_matrix,classification_report, precision_recall_curve
    import numpy as np
    ```

1. Разделите ваши тренировочные и тестовые данные:

    ```python
    X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)
    ```



## Классификатор Linear SVC

Support-Vector classification (SVC) является представителем семейства методов машинного обучения Support-Vector machines (узнайте больше ниже). В этом методе вы можете выбрать «ядро» для определения того, как разделять метки. Параметр «C» относится к «регуляризации», которая регулирует влияние параметров. Ядро может быть одним из [нескольких](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC); здесь мы установили его на 'linear', чтобы использовать линейный SVC. По умолчанию вероятность установлена в 'false'; здесь мы ставим ее в 'true' для получения оценок вероятности. Мы устанавливаем случайное состояние в '0' для перемешивания данных и получения вероятностей.

### Упражнение - примените линейный SVC

Начните с создания массива классификаторов. Вы будете постепенно добавлять в этот массив по мере тестирования.

1. Начните с Linear SVC:

    ```python
    C = 10
    # Создайте различные классификаторы.
    classifiers = {
        'Linear SVC': SVC(kernel='linear', C=C, probability=True,random_state=0)
    }
    ```

2. Обучите свою модель с использованием Linear SVC и выведите отчет:

    ```python
    n_classifiers = len(classifiers)
    
    for index, (name, classifier) in enumerate(classifiers.items()):
        classifier.fit(X_train, np.ravel(y_train))
    
        y_pred = classifier.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        print("Accuracy (test) for %s: %0.1f%% " % (name, accuracy * 100))
        print(classification_report(y_test,y_pred))
    ```

    Результат довольно хороший:

    ```output
    Accuracy (test) for Linear SVC: 78.6% 
                  precision    recall  f1-score   support
    
         chinese       0.71      0.67      0.69       242
          indian       0.88      0.86      0.87       234
        japanese       0.79      0.74      0.76       254
          korean       0.85      0.81      0.83       242
            thai       0.71      0.86      0.78       227
    
        accuracy                           0.79      1199
       macro avg       0.79      0.79      0.79      1199
    weighted avg       0.79      0.79      0.79      1199
    ```



## Классификатор K-Neighbors

K-Neighbors принадлежит к семейству методов «neighbors» машинного обучения, которые можно использовать как для контролируемого, так и для неконтролируемого обучения. В этом методе создается заранее определенное количество точек, и данные собираются вокруг этих точек так, чтобы можно было предсказывать обобщенные метки для данных.

### Упражнение - примените классификатор K-Neighbors

Предыдущий классификатор был хорош и хорошо работал с данными, но возможно, мы можем получить лучшую точность. Попробуйте классификатор K-Neighbors.

1. Добавьте строку в ваш массив классификаторов (добавьте запятую после элемента Linear SVC):

    ```python
    'KNN classifier': KNeighborsClassifier(C),
    ```

    Результат немного хуже:

    ```output
    Accuracy (test) for KNN classifier: 73.8% 
                  precision    recall  f1-score   support
    
         chinese       0.64      0.67      0.66       242
          indian       0.86      0.78      0.82       234
        japanese       0.66      0.83      0.74       254
          korean       0.94      0.58      0.72       242
            thai       0.71      0.82      0.76       227
    
        accuracy                           0.74      1199
       macro avg       0.76      0.74      0.74      1199
    weighted avg       0.76      0.74      0.74      1199
    ```

    ✅ Узнайте о [K-Neighbors](https://scikit-learn.org/stable/modules/neighbors.html#neighbors)



## Классификатор Support Vector

Классификаторы Support-Vector относятся к семейству методов [Support-Vector Machine](https://wikipedia.org/wiki/Support-vector_machine) для задач классификации и регрессии. SVM «отображает тренировочные примеры в точки в пространстве» для максимизации расстояния между двумя категориями. Последующие данные отображаются в это пространство, чтобы предсказать их категорию.

### Упражнение - примените классификатор Support Vector

Попробуем добиться немного лучшей точности с помощью классификатора Support Vector.

1. Добавьте запятую после элемента K-Neighbors, а затем добавьте эту строку:

    ```python
    'SVC': SVC(),
    ```

    Результат довольно хороший!

    ```output
    Accuracy (test) for SVC: 83.2% 
                  precision    recall  f1-score   support
    
         chinese       0.79      0.74      0.76       242
          indian       0.88      0.90      0.89       234
        japanese       0.87      0.81      0.84       254
          korean       0.91      0.82      0.86       242
            thai       0.74      0.90      0.81       227
    
        accuracy                           0.83      1199
       macro avg       0.84      0.83      0.83      1199
    weighted avg       0.84      0.83      0.83      1199
    ```

    ✅ Узнайте о [Support-Vectors](https://scikit-learn.org/stable/modules/svm.html#svm)



## Ансамблевые классификаторы

Давайте пройдем путь до самого конца, хотя предыдущий тест был довольно хорошим. Попробуем «Ансамблевые классификаторы», а именно Random Forest и AdaBoost:



In [ ]:
'RFST': RandomForestClassifier(n_estimators=100),
  'ADA': AdaBoostClassifier(n_estimators=100)

Результат очень хороший, особенно для Random Forest:

```output
Accuracy (test) for RFST: 84.5% 
              precision    recall  f1-score   support

     chinese       0.80      0.77      0.78       242
      indian       0.89      0.92      0.90       234
    japanese       0.86      0.84      0.85       254
      korean       0.88      0.83      0.85       242
        thai       0.80      0.87      0.83       227

    accuracy                           0.84      1199
   macro avg       0.85      0.85      0.84      1199
weighted avg       0.85      0.84      0.84      1199

Accuracy (test) for ADA: 72.4% 
              precision    recall  f1-score   support

     chinese       0.64      0.49      0.56       242
      indian       0.91      0.83      0.87       234
    japanese       0.68      0.69      0.69       254
      korean       0.73      0.79      0.76       242
        thai       0.67      0.83      0.74       227

    accuracy                           0.72      1199
   macro avg       0.73      0.73      0.72      1199
weighted avg       0.73      0.72      0.72      1199
```

✅ Узнайте об [Ансамблевых классификаторах](https://scikit-learn.org/stable/modules/ensemble.html)

Этот метод машинного обучения «комбинирует предсказания нескольких базовых оценщиков», чтобы улучшить качество модели. В нашем примере мы использовали случайные деревья и AdaBoost.

- [Random Forest](https://scikit-learn.org/stable/modules/ensemble.html#forest), метод усреднения, строит «лес» из «решающих деревьев», наполненный случайностью, чтобы избежать переобучения. Параметр n_estimators установлен на количество деревьев.

- [AdaBoost](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) подгоняет классификатор к набору данных, а затем снова подгоняет копии этого классификатора к тому же набору данных. Он фокусируется на весах неправильно классифицированных объектов и корректирует подгонку для следующего классификатора, чтобы исправить ошибки.



## Вызов

Каждая из этих техник имеет множество параметров, которые вы можете настроить. Изучите параметры по умолчанию каждого и подумайте, что изменение этих параметров будет означать для качества модели.



## Обзор и самостоятельное обучение

В этих уроках много терминов, поэтому найдите минуту, чтобы просмотреть [этот список](https://learn.microsoft.com/dotnet/machine-learning/resources/glossary?WT.mc_id=academic-77952-leestott) полезной терминологии!



## Задание

[Игра с параметрами](https://github.com/microsoft/ML-For-Beginners/blob/main/4-Classification/3-Classifiers-2/assignment.md)



---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/4-Classification/3-Classifiers-2)
- [Учебник ML (ШАД): обучение с учителем](https://contest.yandex.ru/tracks/ml/supervised-learning/introduction)
- [Учебник ML (ШАД): метрики и кросс-валидация](https://contest.yandex.ru/tracks/ml/model-evaluation/introduction)
- [Математика для анализа данных: теория вероятностей](https://contest.yandex.ru/tracks/math/probability-theory/chapter-overview)